# Geometric Field Modulation for Ternary Parameter Golf

Phase 0: Measure ternary quantization error structure (DECISION GATE).
If structure exists → proceed with geometric field G experiments.
If no structure → stop and write up negative result.

Based on the ternary submission by Ciprian-Florin Ifrim (1.1570 BPB).

## 1. Setup

In [ ]:
import shutil, os, glob

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'

OFFICIAL = '/content/parameter-golf'
AUX = '/content/parameter-golf-aux'
TERNARY_SRC = f'{OFFICIAL}/records/track_10min_16mb/2026-03-24_74M_Ternary_UNet_FP8_10L_8192BPE_YaRN_NeoMuon'

# Clone repos
%cd /content
if not os.path.exists(OFFICIAL):
    !git clone --depth 1 https://github.com/openai/parameter-golf.git
else:
    %cd {OFFICIAL}
    !git pull
if not os.path.exists(AUX):
    !git clone https://github.com/jamesconde/parameter-golf-aux.git
else:
    %cd {AUX}
    !git pull

!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard

# Copy geometric field scripts to working area
WORK = '/content/ternary_work'
os.makedirs(WORK, exist_ok=True)

# Copy ternary training script
shutil.copy2(f'{TERNARY_SRC}/train_gpt_cuda_ternary.py', f'{WORK}/train_gpt_cuda_ternary.py')
shutil.copy2(f'{TERNARY_SRC}/fineweb_8192_bpe.model', f'{WORK}/fineweb_8192_bpe.model')
shutil.copy2(f'{TERNARY_SRC}/fineweb_8192_bpe.vocab', f'{WORK}/fineweb_8192_bpe.vocab')

# Copy our analysis scripts
os.makedirs(f'{WORK}/geometric_field', exist_ok=True)
for f in glob.glob(f'{AUX}/geometric_field/*.py'):
    shutil.copy2(f, f'{WORK}/geometric_field/{os.path.basename(f)}')

# Link data from Drive or download
DRIVE_DATA_8K = f'{DRIVE_DIR}/data/datasets/fineweb10B_sp8192'
LOCAL_DATA_8K = f'{WORK}/data/datasets/fineweb10B_sp8192'
os.makedirs(f'{WORK}/data/datasets', exist_ok=True)
os.makedirs(f'{WORK}/data/tokenizers', exist_ok=True)

# Symlink tokenizer
tok_dst = f'{WORK}/data/tokenizers/fineweb_8192_bpe.model'
if not os.path.exists(tok_dst):
    shutil.copy2(f'{WORK}/fineweb_8192_bpe.model', tok_dst)

print(f'Working directory: {WORK}')
print(f'Ternary script: {WORK}/train_gpt_cuda_ternary.py')

In [ ]:
# Download 8192 BPE data (different tokenizer than sp1024)
%cd {OFFICIAL}

if not os.path.exists(f'{OFFICIAL}/data/datasets/fineweb10B_sp8192/fineweb_val_000000.bin'):
    !python3 data/cached_challenge_fineweb.py --variant sp8192 --train-shards 1

# Link to working directory
src_8k = f'{OFFICIAL}/data/datasets/fineweb10B_sp8192'
if os.path.exists(src_8k) and not os.path.exists(LOCAL_DATA_8K):
    os.symlink(src_8k, LOCAL_DATA_8K)
    print(f'Linked {src_8k} → {LOCAL_DATA_8K}')

import glob
train_files = glob.glob(f'{LOCAL_DATA_8K}/fineweb_train_*.bin')
val_files = glob.glob(f'{LOCAL_DATA_8K}/fineweb_val_*.bin')
print(f'Train shards: {len(train_files)}, Val shards: {len(val_files)}')

%%time
%cd {WORK}

import os

# Patch ternary script for non-Hopper GPU (FA fallback + compile flag)
!python3 geometric_field/patch_ternary.py train_gpt_cuda_ternary.py

# Check GPU
import torch
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({vram_gb:.0f}GB)')

if os.path.exists('final_model.pt') or os.path.exists(f'{DRIVE_DIR}/ternary/final_model.pt'):
    if os.path.exists(f'{DRIVE_DIR}/ternary/final_model.pt') and not os.path.exists('final_model.pt'):
        import shutil
        shutil.copy2(f'{DRIVE_DIR}/ternary/final_model.pt', 'final_model.pt')
    print(f'Ternary model already exists ({os.path.getsize("final_model.pt")/1e6:.1f} MB)')
else:
    print('Training ternary baseline (~1 hour)...')
    !RUN_ID=ternary_baseline \
     DATA_PATH=./data/datasets/fineweb10B_sp8192 \
     TOKENIZER_PATH=./data/tokenizers/fineweb_8192_bpe.model \
     VOCAB_SIZE=8192 \
     NUM_LAYERS=10 MODEL_DIM=768 NUM_HEADS=8 NUM_KV_HEADS=4 MLP_MULT=4 \
     EMBED_DIM=254 \
     ACTIVATION=relu2 \
     TRAIN_BATCH_TOKENS=524288 TRAIN_SEQ_LEN=1024 \
     ITERATIONS=10000 MAX_WALLCLOCK_SECONDS=3600 \
     WARMUP_STEPS=5 VAL_LOSS_EVERY=500 \
     USE_COMPILE=0 \
     SEED=42 \
     python3 train_gpt_cuda_ternary.py

In [ ]:
%%time
%cd {WORK}

import os
if os.path.exists('final_model.pt') or os.path.exists(f'{DRIVE_DIR}/ternary/final_model.pt'):
    if os.path.exists(f'{DRIVE_DIR}/ternary/final_model.pt') and not os.path.exists('final_model.pt'):
        import shutil
        shutil.copy2(f'{DRIVE_DIR}/ternary/final_model.pt', 'final_model.pt')
    print(f'Ternary model already exists ({os.path.getsize("final_model.pt")/1e6:.1f} MB)')
else:
    print('Training ternary baseline (~1 hour)...')
    # Note: may need FA fallback patches for non-Hopper GPUs
    !RUN_ID=ternary_baseline \
     DATA_PATH=./data/datasets/fineweb10B_sp8192 \
     TOKENIZER_PATH=./data/tokenizers/fineweb_8192_bpe.model \
     VOCAB_SIZE=8192 \
     NUM_LAYERS=10 MODEL_DIM=768 NUM_HEADS=8 NUM_KV_HEADS=4 MLP_MULT=4 \
     EMBED_DIM=254 \
     ACTIVATION=relu2 \
     TRAIN_BATCH_TOKENS=524288 TRAIN_SEQ_LEN=1024 \
     ITERATIONS=10000 MAX_WALLCLOCK_SECONDS=3600 \
     WARMUP_STEPS=5 VAL_LOSS_EVERY=500 \
     COMPILE_MODE=default USE_COMPILE=0 \
     SEED=42 \
     python3 train_gpt_cuda_ternary.py

In [ ]:
# Save checkpoint to Drive
import shutil
for f in ['final_model.pt']:
    if os.path.exists(f):
        os.makedirs(f'{DRIVE_DIR}/ternary', exist_ok=True)
        shutil.copy2(f, f'{DRIVE_DIR}/ternary/{f}')
        print(f'Saved {f} → Drive')

## 3. Phase 0: Quantization Error Structure Analysis (DECISION GATE)

This determines whether geometric field modulation can help.
If no spatial structure → STOP and write up negative result.

In [ ]:
%cd {WORK}

!python3 geometric_field/phase0_analysis.py \
    --checkpoint final_model.pt \
    --model-dim 768 --num-heads 8 --num-kv-heads 4 --mlp-mult 4 --num-layers 10 \
    --group-size 128 \
    --output geometric_field/phase0_results.json

In [ ]:
# Save Phase 0 results to Drive
import shutil, json
for f in ['geometric_field/phase0_results.json', 'geometric_field/phase0_results_profiles.pt']:
    if os.path.exists(f):
        os.makedirs(f'{DRIVE_DIR}/ternary/results', exist_ok=True)
        shutil.copy2(f, f'{DRIVE_DIR}/ternary/results/{os.path.basename(f)}')

# Display classification
if os.path.exists('geometric_field/phase0_results.json'):
    with open('geometric_field/phase0_results.json') as f:
        results = json.load(f)
    c = results['classification']
    print(f"\nScenario: {c['scenario']}")
    print(f"Proceed: {c['proceed']}")
    print(f"Row structure: {c['mean_row_structure']:.2f}")
    print(f"Col structure: {c['mean_col_structure']:.2f}")

## 4. Next Steps (only if Phase 0 says PROCEED)

If Scenario C, D, or E: run signal computation and G experiments.
If Scenario A or B: write up negative result.

In [ ]:
# Placeholder for Experiment 1-5 (implement only if Phase 0 passes)
print('Phase 0 must pass before proceeding.')
print('Check the scenario classification above.')
print('If PROCEED: continue with signal computation.')
print('If STOP: valuable negative result for the submission writeup.')